In [ ]:
#| hide
from fastermodels import *

# fastermodels

> Load, evaluate and publish models optimized by fasterai

A model optimized by [fasterai](https://github.com/FasterAI-Labs/fasterai) no longer has the shapes its
factory builds — a pruned ResNet has fewer channels than `torchvision.models.resnet18` gives you, and a
checkpoint alone will not load. `fastermodels` publishes the missing half: the factory, the rebuild spec of
the layers that changed, and the weights, in one artifact that reloads anywhere with
`FasterModel.from_pretrained`.

It also carries what a reader needs in order to trust the artifact:

- **`FasterModel`** — rebuild the architecture from its spec, load the weights strictly, push to the Hub;
  **`load`** brings back whichever form a repo publishes — weights, TorchScript or ONNX
- **eval** — the four criteria a card carries: top-1 (per-image correctness, Wilson intervals, paired deltas
  with an exact McNemar p), size, memory (peak live activations) and MACs
- **card** — a model card written from measured values only, and a reader that flags what it should not say
- **gate** — the ten conditions an artifact meets before it is published

## Install

```sh
pip install fastermodels
```

## How to use

Publish a model you just optimized:

```python
from fastermodels import FasterModel

fm = FasterModel.wrap(pruned, 'torchvision.models.resnet18', {'num_classes': 10, 'weights': None},
                      recipe={'prune': 'ratio 0.3, local, round_to 8'})
fm.save_pretrained('artifact')            # config.json + model.safetensors
```

Load it back and measure it:

```python
from fastermodels import load, correct_vector, wilson, paired_delta, params, macs, peak_activation_bytes

fm = load('artifact')                          # safetensors, else TorchScript, else the ONNX path
opt = correct_vector(fm, valid_dl)
k, n = int(opt.sum()), opt.size
wilson(k, n)                                   # the interval that belongs next to k/n
paired_delta(correct_vector(source, valid_dl), opt)
params(fm), macs(fm, sample), peak_activation_bytes(fm, sample)   # size, compute, memory at batch 1
```

Write its card and run the gate before publishing:

```python
from fastermodels import render_card, check_card, run_gate, gate_passed

open('artifact/README.md', 'w').write(render_card(meta))
check_card(open('artifact/README.md').read())          # [] when the card is clean
gate_passed(run_gate('artifact', manifest))
```

## Documentation

The API pages are at [FasterAI-Labs.github.io/fastermodels](https://FasterAI-Labs.github.io/fastermodels/):
[Model](00_model.html), [Eval](01_eval.html), [Card](02_card.html), [Gate](03_gate.html).